# Clean Raw Weather Data

Notebook làm sạch dữ liệu raw từ `data/raw/Hanoi.csv`, chọn feature bằng Random Forest với TimeSeriesSplit, rồi lưu dữ liệu cho model vào `data/processed/Hanoi_final.csv`.


In [1]:
from pathlib import Path
import sys
import hashlib
import json

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

In [2]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data/raw/Hanoi.csv").exists() and (PROJECT_ROOT.parent / "data/raw/Hanoi.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
from src.weather_data import WEATHER_COLUMNS, METADATA_COLUMNS, validate_hourly_frame, build_forecast_dataset

RAW_PATH = PROJECT_ROOT / "data/raw/Hanoi.csv"
OUTPUT_PATH = PROJECT_ROOT / "data/processed/Hanoi_final.csv"
FEATURE_SELECTION_PATH = PROJECT_ROOT / "data_processing/feature_selection.json"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

TARGET_COLUMN = "temperature_2m"
HORIZON_HOURS = 1
CV_SPLITS = 5
TEST_SIZE = 0.2
RANDOM_STATE = 42
MAE_TOLERANCE_PCT = 1.0

print(RAW_PATH)
print(OUTPUT_PATH)


/Users/leducanhtuan/Downloads/Weather-Forecasting-ystem-main/data/raw/Hanoi.csv
/Users/leducanhtuan/Downloads/Weather-Forecasting-ystem-main/data/processed/Hanoi_final.csv


## 1. Đọc dữ liệu raw

In [3]:
df = pd.read_csv(RAW_PATH)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
df.head()

Rows: 140,256
Columns: 34


,time,temperature_2m,relative_humidity_2m,dew_point_2m,apparent_temperature,precipitation,rain,snowfall,pressure_msl,surface_pressure,...,soil_temperature_100_to_255cm,soil_moisture_0_to_7cm,soil_moisture_7_to_28cm,soil_moisture_28_to_100cm,soil_moisture_100_to_255cm,weather_code,is_day,city,latitude,longitude
0,2010-01-01 00:00:00,16.7,95,16.0,18.4,0.2,0.2,0.0,1015.6,1013.3,...,23.6,0.305,0.265,0.292,0.38,51,0,Hanoi,21.0285,105.8542
1,2010-01-01 01:00:00,16.8,95,16.0,18.4,0.2,0.2,0.0,1015.5,1013.2,...,23.6,0.307,0.265,0.292,0.38,51,0,Hanoi,21.0285,105.8542
2,2010-01-01 02:00:00,16.8,95,16.0,18.2,0.3,0.3,0.0,1014.9,1012.6,...,23.6,0.309,0.265,0.292,0.38,51,0,Hanoi,21.0285,105.8542
3,2010-01-01 03:00:00,16.5,96,15.9,17.8,0.3,0.3,0.0,1015.0,1012.7,...,23.6,0.310,0.265,0.292,0.38,51,0,Hanoi,21.0285,105.8542
4,2010-01-01 04:00:00,16.5,96,15.8,17.6,0.2,0.2,0.0,1014.7,1012.4,...,23.6,0.313,0.266,0.292,0.38,51,0,Hanoi,21.0285,105.8542


## 2. Chuẩn hoá thời gian và xoá trùng

In [4]:
df["time"] = pd.to_datetime(df["time"], errors="raise")
duplicate_rows = int(df.duplicated().sum())
df = df.drop_duplicates().sort_values("time").reset_index(drop=True)
if df["time"].isna().any() or df["time"].duplicated().any():
    raise ValueError("Invalid timestamps or conflicting rows for the same hour.")
print(f"Exact duplicate rows removed: {duplicate_rows:,}")
print(f"Time range: {df['time'].min()} -> {df['time'].max()}")


Exact duplicate rows removed: 0
Time range: 2010-01-01 00:00:00 -> 2025-12-31 23:00:00


## 3. Chuẩn hoá kiểu dữ liệu

In [5]:
for column in df.columns:
    if column not in ["time", "city"]:
        df[column] = pd.to_numeric(df[column], errors="raise")
df.dtypes


time                             datetime64[ns]
temperature_2m                          float64
relative_humidity_2m                      int64
dew_point_2m                            float64
apparent_temperature                    float64
precipitation                           float64
rain                                    float64
snowfall                                float64
pressure_msl                            float64
surface_pressure                        float64
cloud_cover                               int64
cloud_cover_low                           int64
cloud_cover_mid                           int64
cloud_cover_high                          int64
wind_speed_10m                          float64
wind_speed_100m                         float64
wind_direction_10m                        int64
wind_direction_100m                       int64
wind_gusts_10m                          float64
et0_fao_evapotranspiration              float64
vapour_pressure_deficit                 

## 4. Xử lý missing values

In [6]:
missing_before = df.isna().sum().sort_values(ascending=False)
missing_before[missing_before > 0].head(30)

Series([], dtype: int64)

In [7]:
# Không tạo nhãn giả và không lấy giá trị tương lai để nội suy.
if df[TARGET_COLUMN].isna().any():
    raise ValueError("Missing target measurements; retrieve complete archive data first.")
feature_columns_to_fill = [c for c in df.columns if c not in ["time", TARGET_COLUMN]]
df[feature_columns_to_fill] = df[feature_columns_to_fill].ffill(limit=3)
validate_hourly_frame(df, [*WEATHER_COLUMNS, *METADATA_COLUMNS])
print(f"Total missing after causal cleaning: {int(df.isna().sum().sum()):,}")


Total missing after causal cleaning: 0


## 5. Kiểm tra khoảng thời gian

In [8]:
validate_hourly_frame(df, WEATHER_COLUMNS)
print("Verified: unique timestamps, complete hourly timeline, finite values.")


Verified: unique timestamps, complete hourly timeline, finite values.


## 6. Feature selection với TimeSeriesSplit

Mục tiêu là dự báo `temperature_2m` một giờ sau. Chỉ dùng `cloud_cover` để biểu diễn độ phủ mây tổng thể; ba cột mây theo tầng không tham gia chọn biến. Giữ 20% thời gian cuối làm holdout; chỉ 80% đầu được dùng để chọn feature. Trong phần này, `TimeSeriesSplit` 5 fold với `gap=1` giờ xếp hạng feature bằng Random Forest trên train của từng fold, rồi so MAE validation của mọi tập top-k trong các biến ứng viên. Chọn tập nhỏ nhất có MAE trung bình không cao hơn mức thấp nhất quá 1%; sau đó chỉ dùng holdout để kiểm tra kết quả.


In [9]:
def build_feature_selection_dataset(
    dataframe: pd.DataFrame,
    target_column: str,
    horizon_hours: int,
) -> tuple[pd.DataFrame, list[str], str, list[str]]:
    working_df = dataframe.copy()
    label_column = f"{target_column}_future_{horizon_hours}h"

    numeric_columns = working_df.select_dtypes(include=[np.number]).columns.tolist()
    excluded_columns = {
        label_column, "latitude", "longitude",
        "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high",
    }
    candidate_features = [column for column in numeric_columns if column not in excluded_columns]

    # Xác định cột hằng chỉ từ phần train; holdout không tham gia selection.
    train_end = int(len(working_df) * (1 - TEST_SIZE)) - horizon_hours
    constant_features = [
        column for column in candidate_features
        if working_df.iloc[:train_end][column].nunique(dropna=False) <= 1
    ]
    candidate_features = [column for column in candidate_features if column not in constant_features]

    selection_df, label_column = build_forecast_dataset(
        working_df, candidate_features, target_column, horizon_hours,
    )
    return selection_df, candidate_features, label_column, constant_features


selection_df, candidate_features, label_column, constant_features = build_feature_selection_dataset(
    dataframe=df,
    target_column=TARGET_COLUMN,
    horizon_hours=HORIZON_HOURS,
)

print(f"Candidate features: {len(candidate_features):,}")
print(f"Rows for feature selection: {len(selection_df):,}")
print(f"Constant features removed: {constant_features}")
selection_df[["time", *candidate_features[:8], label_column]].head()


Candidate features: 26
Rows for feature selection: 140,255
Constant features removed: ['snowfall']


,time,temperature_2m,relative_humidity_2m,dew_point_2m,apparent_temperature,precipitation,rain,pressure_msl,surface_pressure,temperature_2m_future_1h
0,2010-01-01 00:00:00,16.7,95,16.0,18.4,0.2,0.2,1015.6,1013.3,16.8
1,2010-01-01 01:00:00,16.8,95,16.0,18.4,0.2,0.2,1015.5,1013.2,16.8
2,2010-01-01 02:00:00,16.8,95,16.0,18.2,0.3,0.3,1014.9,1012.6,16.5
3,2010-01-01 03:00:00,16.5,96,15.9,17.8,0.3,0.3,1015.0,1012.7,16.5
4,2010-01-01 04:00:00,16.5,96,15.8,17.6,0.2,0.2,1014.7,1012.4,16.6


In [10]:
holdout_start = df["time"].iloc[int(len(df) * (1 - TEST_SIZE))]
selection_train = selection_df.loc[selection_df["label_time"] < holdout_start].copy()
selection_holdout = selection_df.loc[selection_df["label_time"] >= holdout_start].copy()
assert selection_train["label_time"].max() < selection_holdout["label_time"].min()

tscv = TimeSeriesSplit(n_splits=CV_SPLITS, gap=HORIZON_HOURS)
max_k = len(candidate_features)
k_candidates = list(range(1, max_k + 1))

fold_metrics = []
fold_importances = []
fold_data = []

for fold, (train_index, valid_index) in enumerate(tscv.split(selection_train), start=1):
    X_train = selection_train.iloc[train_index][candidate_features]
    X_valid = selection_train.iloc[valid_index][candidate_features]
    y_train = selection_train.iloc[train_index][label_column]
    y_valid = selection_train.iloc[valid_index][label_column]

    selector_model = RandomForestRegressor(
        n_estimators=60,
        max_depth=12,
        min_samples_leaf=2,
        random_state=RANDOM_STATE + fold,
        n_jobs=-1,
    )
    selector_model.fit(X_train, y_train)
    importance = selector_model.feature_importances_
    ranked_features = [candidate_features[i] for i in np.argsort(-importance)]
    fold_importances.append(importance)
    fold_data.append((fold, X_train, X_valid, y_train, y_valid, ranked_features))

    fold_metrics.append({
        "fold": fold,
        "train_start": selection_train.iloc[train_index[0]]["time"],
        "train_end": selection_train.iloc[train_index[-1]]["time"],
        "valid_start": selection_train.iloc[valid_index[0]]["time"],
        "valid_end": selection_train.iloc[valid_index[-1]]["time"],
    })
    print(f"Ranked {len(candidate_features)} features on fold {fold}/{CV_SPLITS}.")


def score_subset(k: int) -> list[dict]:
    rows = []
    for fold, X_train, X_valid, y_train, y_valid, ranked_features in fold_data:
        columns = ranked_features[:k]
        model = RandomForestRegressor(
            n_estimators=60,
            max_depth=12,
            min_samples_leaf=2,
            random_state=RANDOM_STATE + fold,
            n_jobs=-1,
        )
        model.fit(X_train[columns], y_train)
        predictions = model.predict(X_valid[columns])
        rows.append({
            "k": k,
            "fold": fold,
            "MAE": mean_absolute_error(y_valid, predictions),
            "RMSE": np.sqrt(mean_squared_error(y_valid, predictions)),
            "R2": r2_score(y_valid, predictions),
        })
    return rows


subset_results = []
for k in k_candidates:
    subset_results.extend(score_subset(k))
    print(f"Checked top-{k} features.")

subset_cv_results = pd.DataFrame(subset_results)
cv_summary = subset_cv_results.groupby("k", as_index=False).agg(
    MAE_mean=("MAE", "mean"),
    MAE_std=("MAE", "std"),
    RMSE_mean=("RMSE", "mean"),
    RMSE_std=("RMSE", "std"),
    R2_mean=("R2", "mean"),
)
cv_summary = cv_summary.sort_values(["MAE_mean", "k"]).reset_index(drop=True)
best_mae = cv_summary["MAE_mean"].min()
eligible_k = cv_summary.loc[
    cv_summary["MAE_mean"] <= best_mae * (1 + MAE_TOLERANCE_PCT / 100), "k"
]
best_k = int(eligible_k.min())

print(f"Smallest k within {MAE_TOLERANCE_PCT:g}% of best CV MAE: {best_k}")
print(f"Train period: {selection_train['time'].min()} -> {selection_train['time'].max()}")
print(f"Holdout period: {selection_holdout['time'].min()} -> {selection_holdout['time'].max()}")
cv_summary.head(20)


Ranked 26 features on fold 1/5.


Ranked 26 features on fold 2/5.


Ranked 26 features on fold 3/5.


Ranked 26 features on fold 4/5.


Ranked 26 features on fold 5/5.


Checked top-1 features.


... (22 outputs left out)

Checked top-24 features.


Checked top-25 features.


Checked top-26 features.
Smallest k within 1% of best CV MAE: 9
Train period: 2010-01-01 00:00:00 -> 2022-10-20 02:00:00
Holdout period: 2022-10-20 03:00:00 -> 2025-12-31 22:00:00


,k,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean
0,11,0.470434,0.040925,0.695510,0.071303,0.984911
1,20,0.470603,0.043872,0.693455,0.075330,0.984995
2,14,0.470702,0.041444,0.694252,0.071030,0.984965
3,18,0.470723,0.043319,0.693985,0.074514,0.984970
4,26,0.470727,0.042212,0.693054,0.072869,0.985014
5,24,0.470802,0.042400,0.693314,0.073206,0.985003
6,25,0.470866,0.042120,0.693481,0.072834,0.984996
7,17,0.470868,0.041999,0.694351,0.072674,0.984955
8,15,0.470880,0.041569,0.694415,0.070874,0.984958
9,19,0.470920,0.043416,0.694397,0.074669,0.984953


In [11]:
importance_array = np.vstack(fold_importances)
feature_scores = pd.DataFrame({
    "feature": candidate_features,
    "importance_mean": importance_array.mean(axis=0),
    "importance_std": importance_array.std(axis=0),
})
feature_scores = feature_scores.sort_values("importance_mean", ascending=False).reset_index(drop=True)
feature_scores["rank"] = np.arange(1, len(feature_scores) + 1)
feature_scores["selected"] = feature_scores["rank"] <= best_k
selected_features = feature_scores.loc[feature_scores["selected"], "feature"].tolist()

# Holdout chỉ đánh giá lựa chọn đã cố định; không dùng để chọn lại k.
holdout_rows = []
for name, columns in [("selected", selected_features), ("all_candidates", candidate_features)]:
    model = RandomForestRegressor(
        n_estimators=60,
        max_depth=12,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    model.fit(selection_train[columns], selection_train[label_column])
    predictions = model.predict(selection_holdout[columns])
    holdout_rows.append({
        "feature_set": name,
        "k": len(columns),
        "MAE": mean_absolute_error(selection_holdout[label_column], predictions),
        "RMSE": np.sqrt(mean_squared_error(selection_holdout[label_column], predictions)),
        "R2": r2_score(selection_holdout[label_column], predictions),
    })
holdout_results = pd.DataFrame(holdout_rows)

print(f"Selected features: {len(selected_features):,}")
print(holdout_results.to_string(index=False))
feature_scores.head(30)


Selected features: 9
   feature_set  k      MAE     RMSE       R2
      selected  9 0.444338 0.650627 0.986462
all_candidates 26 0.442963 0.638694 0.986954


,feature,importance_mean,importance_std,rank,selected
0,temperature_2m,0.962779,0.001704,1,True
1,apparent_temperature,0.019149,0.001229,2,True
2,et0_fao_evapotranspiration,0.005233,0.000140,3,True
3,vapour_pressure_deficit,0.001421,0.000063,4,True
4,soil_temperature_0_to_7cm,0.001401,0.000233,5,True
5,soil_temperature_7_to_28cm,0.001145,0.000080,6,True
6,dew_point_2m,0.001063,0.000045,7,True
7,is_day,0.000983,0.000167,8,True
8,cloud_cover,0.000877,0.000030,9,True
9,soil_moisture_0_to_7cm,0.000693,0.000067,10,False


In [12]:
selected_features_df = feature_scores.loc[
    feature_scores["selected"],
    ["rank", "feature", "importance_mean", "importance_std"],
].copy()

# Một báo cáo chứa danh sách đầu vào cho model và toàn bộ bảng đánh giá.
selection_report = {
    "target": TARGET_COLUMN,
    "horizon_hours": HORIZON_HOURS,
    "cv_splits": CV_SPLITS,
    "holdout_fraction": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "mae_tolerance_pct": MAE_TOLERANCE_PCT,
    "holdout_start": str(holdout_start),
    "selection_train_label_end": str(selection_train["label_time"].max()),
    "raw_sha256": hashlib.sha256(RAW_PATH.read_bytes()).hexdigest(),
    "selected_features": selected_features_df["feature"].tolist(),
    "feature_scores": feature_scores.to_dict(orient="records"),
    "subset_cv_scores": cv_summary.to_dict(orient="records"),
    "holdout_scores": holdout_results.to_dict(orient="records"),
}
FEATURE_SELECTION_PATH.write_text(
    json.dumps(selection_report, indent=2, ensure_ascii=False, allow_nan=False) + "\n",
    encoding="utf-8",
)
print(f"Saved feature selection report: {FEATURE_SELECTION_PATH}")
selected_features_df


Saved feature selection report: /Users/leducanhtuan/Downloads/Weather-Forecasting-ystem-main/data_processing/feature_selection.json


,rank,feature,importance_mean,importance_std
0,1,temperature_2m,0.962779,0.001704
1,2,apparent_temperature,0.019149,0.001229
2,3,et0_fao_evapotranspiration,0.005233,0.000140
3,4,vapour_pressure_deficit,0.001421,0.000063
4,5,soil_temperature_0_to_7cm,0.001401,0.000233
5,6,soil_temperature_7_to_28cm,0.001145,0.000080
6,7,dew_point_2m,0.001063,0.000045
7,8,is_day,0.000983,0.000167
8,9,cloud_cover,0.000877,0.000030


## 7. Áp dụng selected features và lưu dữ liệu sạch

Giữ các feature đã chọn cùng thông tin thời gian/vị trí và lưu thẳng vào `Hanoi_final.csv`. Nhãn nhiệt độ giờ tiếp theo được tạo khi chuẩn bị dữ liệu cho model.


In [13]:
metadata_columns = [
    column for column in ["time", "city", "latitude", "longitude"]
    if column in df.columns
]
clean_columns = list(dict.fromkeys([
    *metadata_columns, TARGET_COLUMN, *selected_features,
]))
removed_columns = [column for column in df.columns if column not in clean_columns]

df = df[clean_columns].copy()
validate_hourly_frame(df, selected_features)
df.to_csv(OUTPUT_PATH, index=False)
selection_report["processed_sha256"] = hashlib.sha256(OUTPUT_PATH.read_bytes()).hexdigest()
FEATURE_SELECTION_PATH.write_text(
    json.dumps(selection_report, indent=2, ensure_ascii=False, allow_nan=False) + "\n",
    encoding="utf-8",
)
print(f"Saved: {OUTPUT_PATH}")
print(f"Rows: {len(df):,}")
print(f"Selected model features: {len(selected_features):,}")
print(f"Columns including metadata: {len(df.columns):,}")
print(f"Removed columns: {removed_columns}")


Saved: /Users/leducanhtuan/Downloads/Weather-Forecasting-ystem-main/data/processed/Hanoi_final.csv
Rows: 140,256
Selected model features: 9
Columns including metadata: 13
Removed columns: ['relative_humidity_2m', 'precipitation', 'rain', 'snowfall', 'pressure_msl', 'surface_pressure', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high', 'wind_speed_10m', 'wind_speed_100m', 'wind_direction_10m', 'wind_direction_100m', 'wind_gusts_10m', 'soil_temperature_28_to_100cm', 'soil_temperature_100_to_255cm', 'soil_moisture_0_to_7cm', 'soil_moisture_7_to_28cm', 'soil_moisture_28_to_100cm', 'soil_moisture_100_to_255cm', 'weather_code']
